# 🎨 NÉT STUDIO — HUẤN LUYỆN AI MÓC KHOÁ ĐỘC BẢN (LoRA + SDXL)
Notebook này được thiết kế để chạy **100% MIỄN PHÍ** trên Google Colab (GPU T4).

### Kết quả đạt được:
* Xuất ra file trọng số: `net_keychain_sdxl_lora.safetensors` (~70MB).
* Khi kết hợp với Prompt hoặc IP-Adapter, AI sẽ luôn tạo ra **móc khoá 2D chuẩn xưởng (có khoen xỏ kim loại trên đỉnh, viền die-cut bo cong, nền trắng tinh khiết)**.

### Bước 1: Kiểm tra GPU & Cài đặt môi trường cần thiết

In [ ]:
!nvidia-smi

# Cài đặt thư viện huấn luyện khuếch tán mới nhất từ Hugging Face
!pip install -q diffusers transformers accelerate peft bitsandbytes xformers torchvision safetensors

### Bước 2: Chuẩn bị thư mục Dataset & Captions

In [ ]:
import os
import json

# Tạo thư mục dữ liệu huấn luyện
os.makedirs('dataset', exist_ok=True)
os.makedirs('output_lora', exist_ok=True)

print('✓ Đã khởi tạo thư mục: /content/dataset và /content/output_lora')
print('Bạn có thể tải ảnh mẫu móc khoá vào thư mục /content/dataset kèm các file .txt mô tả!')

### Bước 3: Tải script huấn luyện chuẩn Diffusers SDXL LoRA

In [ ]:
!wget -q -O train_lora_sdxl.py https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora_sdxl.py
print('✓ Đã tải script train_lora_sdxl.py thành công!')

### Bước 4: Khởi động quá trình Huấn Luyện (Training Loop ~45-60 phút)

In [ ]:
!accelerate launch train_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --train_data_dir="./dataset" \
  --resolution=1024 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1000 \
  --learning_rate=1e-4 \
  --lr_scheduler="cosine" \
  --lr_warmup_steps=100 \
  --mixed_precision="fp16" \
  --rank=32 \
  --output_dir="./output_lora" \
  --seed=42

### Bước 5: Kiểm tra kết quả (Test Inference ngay trong Colab)

In [ ]:
import torch
from diffusers import DiffusionPipeline

pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

# Nạp trọng số LoRA Nét Keychain vừa huấn luyện
pipe.load_lora_weights("./output_lora")

test_prompt = "net_keychain, a 2D flat die-cut acrylic keychain charm of a cool chibi Vietnamese boy with stylish glasses, black t-shirt, direct front view, bold clean black contour die-cut outline, vibrant flat colors, top hanging loop with attached silver metal key ring, isolated on pure solid white #FFFFFF background, no shadows"

image = pipe(prompt=test_prompt, num_inference_steps=30, guidance_scale=7.5).images[0]
image.save("test_net_keychain.png")
display(image)

### Bước 6: Tải file Model `net_keychain_lora.safetensors` về máy tính

In [ ]:
from google.colab import files
import glob

lora_files = glob.glob("./output_lora/*.safetensors")
if lora_files:
    print(f'Đang tải file {lora_files[0]} về máy tính...')
    files.download(lora_files[0])
else:
    print('Chưa tìm thấy file .safetensors. Vui lòng kiểm tra lại bước huấn luyện!')